"""
This code is provided as supplemental material to the publication "Machine learning for small data sets: an exemplary study on the classification of highly complex surface micromorphologies" 
by M. Henkel, M. Sprenger, and O. Lieleg submitted to Materials Today Advances on October 17th, 2025.

"""

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import os
import numpy as np
import logging
from tqdm import tqdm
import yaml
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from torchmetrics.classification import MulticlassCalibrationError

In [ ]:
class Config:
    """Configuration class to store hyperparameters"""
    def __init__(self, config_path='config_CNN.yaml'):
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
            
        self.num_epochs = config.get('num_epochs')
        self.batch_size = config.get('batch_size')
        self.learning_rate = config.get('learning_rate')
        self.num_classes = config.get('num_classes')
        self.image_size = config.get('image_size')
        self.train_ratio = config.get('train_ratio')
        self.val_ratio = config.get('val_ratio')
        self.patience = config.get('patience')
        self.dropout_rate_FC = config.get('dropout_rate_FC')
        self.dropout_rate_CONV = config.get('dropout_rate_CONV')
        self.kernel_size = config.get('kernel_size')
        self.data_augmentation = config.get('data_augmentation')
        self.multiply_training_data = config.get('multiply_training_data')

        



In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        
        padding = (Config().kernel_size - 1) // 2
    
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size= Config().kernel_size, padding=padding),    
            nn.BatchNorm2d(16),                             
            nn.ReLU(),                                      
            nn.MaxPool2d(2, 2),                             
            nn.Dropout2d(Config().dropout_rate_CONV)                              
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size= Config().kernel_size, padding=padding),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(Config().dropout_rate_CONV)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size= Config().kernel_size, padding=padding),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(Config().dropout_rate_CONV)
        )
        
        self.gap = nn.AdaptiveAvgPool2d(1)  
        
        self.fc1 = nn.Sequential(
            nn.Linear(64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(Config().dropout_rate_FC)
        )

        self.fc2 = nn.Sequential(
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(Config().dropout_rate_FC)
        )

        self.fc3 = nn.Linear(128, Config().num_classes)
        
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x)             
        x = x.view(x.size(0), -1)    
        x = self.fc1(x)                 
        x = self.fc2(x)
        features = x
        x = self.fc3(x)
        
        return x, features

In [ ]:
class DatasetManager:
    @staticmethod

    #   Transform dataset
    def get_transforms(training=False):
        if training:
            return transforms.Compose([
                transforms.CenterCrop((419, 559)),  #(height, width) (419, 559)
                transforms.Resize((Config().image_size, Config().image_size)),
                transforms.RandomAffine(
                    degrees=30,  
                    translate=(0.05, 0.05),         
                    scale=(0.90, 1.1),  
                    shear=None 
                ),
                transforms.RandomHorizontalFlip(p=0.3),
                transforms.RandomVerticalFlip(p=0.3),
                transforms.ToTensor(),
            ])
    
        else:
            return transforms.Compose([
                transforms.CenterCrop((419, 559)),  #(height, width) (419, 559)
                transforms.Resize((Config().image_size, Config().image_size)),
                transforms.ToTensor(),
            ])

    @staticmethod
    def load_data(data_path):

        # Load full dataset
        full_dataset = ImageFolderWithPath(root = data_path)
        
        # Calculate splits
        total_size = len(full_dataset)
        train_size = int(Config().train_ratio * total_size)
        val_size = int(Config().val_ratio * total_size)
        test_size = total_size - train_size - val_size
        
        # Split full dataset
        generator = torch.Generator().manual_seed(65)   
        train_dataset, val_dataset, test_dataset =random_split(
            full_dataset, 
            [train_size, val_size, test_size],
            generator=generator
        )
           
        # Create dupclicates for data augmentation  
        if Config().data_augmentation:
            # Create extended indices for data augmentation:
            extended_indices = (
                # Part 1: Guarantee each image appears once
                list(range(len(train_dataset))) +
                
                # Part 2: Add additional random samples:
                # - len(train_dataset) = size of original training set
                # - multiply_training_data - 1 = number of additional copies needed
                # - torch.randint generates random indices between 0 and len(train_dataset)
                # - tolist() converts the tensor of random indices to a Python list
                torch.randint(0, len(train_dataset),
                            (len(train_dataset) * (Config().multiply_training_data - 1),)
                ).tolist()
            )
            
            # Create new training dataset with both original and augmented samples
            # Subset() creates a view of the dataset using the specified indices
            train_dataset = torch.utils.data.Subset(train_dataset, extended_indices)
        
        else:
            pass
    
        # Transform data set
        train_dataset.dataset.transform = DatasetManager.get_transforms(training=True)
        val_dataset.dataset.transform = DatasetManager.get_transforms(training=False)
        test_dataset.dataset.transform = DatasetManager.get_transforms(training=False)

        # Create dataloaders 
        train_loader = DataLoader(
            train_dataset,
            batch_size=Config().batch_size,
            shuffle=True,
            num_workers=0,  
            drop_last=False 
        )
        
        val_loader = DataLoader(
            val_dataset,
            batch_size=Config().batch_size,
            shuffle=True,
            num_workers=0, 
            drop_last=False  
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=Config().batch_size,
            shuffle=True,
            num_workers=0, 
            drop_last=False  
        )

        # Logging
        total_samples = len(train_loader.dataset) + len(val_loader.dataset) + len(test_loader.dataset)

        if Config().data_augmentation:
            original_train_size = len(train_loader.dataset) // Config().multiply_training_data
            logging.info(
                f'Dataset sizes (with data augmentation x{Config().multiply_training_data}):\n'
                f'Training:   {len(train_loader.dataset):5d} samples (original: {original_train_size} samples)\n'
                f'Validation: {len(val_loader.dataset):5d} samples\n'
                f'Test:       {len(test_loader.dataset):5d} samples\n'
                f'Total:      {total_samples:5d} samples'
            )
        else:
            logging.info(
                f'Dataset sizes (no data augmentation):\n'
                f'Training:   {len(train_loader.dataset):5d} samples\n'
                f'Validation: {len(val_loader.dataset):5d} samples\n'
                f'Test:       {len(test_loader.dataset):5d} samples\n'
                f'Total:      {total_samples:5d} samples'
            )
        
        return train_loader, val_loader, test_loader, full_dataset.classes

In [ ]:
class ImageFolderWithPath(torchvision.datasets.ImageFolder):
    """ImageFolder, modified to safe the file paths additionally."""
    def __getitem__(self, index):
        try:
            image, label = super().__getitem__(index)
            path, _ = self.samples[index]
            return image, label, path
            
        except Exception as e:
            # Skip unreadable files (e.g. ._, corrupt images, wrong format)
            return self.__getitem__((index + 1) % len(self.samples))

In [ ]:
class EarlyStopping:
    def __init__(self, min_delta=1e-4):
        self.patience = Config().patience       # How many epochs to wait before stopping
        self.min_delta = min_delta              # Minimum change in accuracy to be considered as improvement
        self.counter = 0                        # Counts epochs without improvement
        self.best_val_acc = 0                   # Stores the best validation accuracy seen so far
        self.early_stop = False       
                  
    def __call__(self, val_acc):
        # Check if current validation accuracy is better than best validation accuracy
        if val_acc > self.best_val_acc + self.min_delta:
            self.best_val_acc = val_acc                        
            self.counter = 0

        else:
            self.counter += 1                   # Increment patience counter
            if self.counter >= self.patience:   # If we've waited long enough
                self.early_stop = True          # Set early stop flag
                return True                     # Signal to stop training
        
        return False  

In [ ]:
class Trainer:
    """Handles model training and evaluation"""
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(model.parameters(), lr=Config().learning_rate, weight_decay=0.05)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='max',           
            factor=0.1,          
            patience=3,           
            verbose=True,
            min_lr=1e-6
        )
        self.early_stopping = EarlyStopping()
        
    def train_epoch(self, train_loader):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        progress_bar = tqdm(train_loader, desc='Training')                      # Progress bar entails training data
        for batch_idx, (images, labels,_) in enumerate(progress_bar):
            images, labels = images.to(self.device), labels.to(self.device)     # Send data to GPU or CPU 
            
            self.optimizer.zero_grad()                                          # Zero all gradients
                                     
            outputs, _ = self.model(images)                                     # Train model on image batch
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)                                # Evaluate output tensor (comprising class probabilities): Ignore max value, get max index
            total += labels.size(0)
            correct += (predicted == labels).sum().item()                        
            
            progress_bar.set_postfix({
                'loss': running_loss/(batch_idx+1),
                'acc': 100.*correct/total
            })
        
        return running_loss/len(train_loader), 100.*correct/total
    
    def validate(self, val_loader):
        self.model.eval()
        correct = 0
        total = 0    
        val_loss = 0.0
            
        with torch.no_grad():
            for images, labels,_ in val_loader:
                images, labels = images.to(self.device), labels.to(self.device) #  Send data to GPU or CPU
                
                outputs, _ = self.model(images)                                 #  Classify validation data
                loss = self.criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
            
        acc = 100.*correct/total   
        avg_loss = val_loss/len(val_loader)

        return acc
        
    def train(self, train_loader, val_loader):
            best_val_acc = 0.
            train_losses = []
            validation_accuracies = []
            best_model_state = None
            best_epoch = 0
            learning_rates = []  

            # Create directory for saved models if it doesn't exist
            save_dir = 'saved_models'
            os.makedirs(save_dir, exist_ok=True)

            # Train model for a set number of epochs
            for epoch in range(Config().num_epochs):
                current_lr = self.optimizer.param_groups[0]['lr']
                learning_rates.append(current_lr)
                
                logging.info(f'\nEpoch {epoch+1}/{Config().num_epochs}')
                logging.info(f'Current learning rate: {current_lr:.6f}')
                
                train_loss, train_acc = self.train_epoch(train_loader)  # Training
                val_acc = self.validate(val_loader)                     # Validation
                
                train_losses.append(train_loss)
                validation_accuracies.append(val_acc)
                
                logging.info(f'Training Loss & Acc.: {train_loss:.4f} Acc: {train_acc:.2f}%')
                logging.info(f'Validation Acc.: {val_acc:.2f}%')
                
                # Control learning rate 
                
                self.scheduler.step(val_acc)
                if current_lr != self.optimizer.param_groups[0]['lr']:
                    logging.info(f'Learning rate reduced to: {self.optimizer.param_groups[0]["lr"]:.6f}')

                    checkpoint_path = os.path.join(save_dir, 'best_model.pth')

                    #Load best previous model
                    if os.path.exists(checkpoint_path):
                        checkpoint = torch.load(checkpoint_path, map_location=self.device)
                        self.model.load_state_dict(checkpoint['model_state_dict'])
                        logging.info('Loaded best model state from checkpoint file')
             
                # Save best model and features
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_epoch = epoch
                    
                    # Save best model checkpoint
                    checkpoint_path = os.path.join(save_dir, 'best_model.pth')
                    self.save_checkpoint(checkpoint_path, epoch, train_loss, val_acc)
                    
                    # Save feature extractor (for few-shot learning)
                    feature_extractor = self.create_feature_extractor()
                    feature_extractor_path = os.path.join(save_dir, 'feature_extractor.pth')
                    torch.save(feature_extractor.state_dict(), feature_extractor_path)
                    logging.info("Saved feature extractor")
                    
                # Early stopping check
                if self.early_stopping(val_acc):
                    logging.info("Early stopping triggered!")
                    if os.path.exists(checkpoint_path):
                        checkpoint = torch.load(checkpoint_path, map_location=self.device)
                        self.model.load_state_dict(checkpoint['model_state_dict'])
                        logging.info('Loaded best model state from checkpoint file')
                    break

            # Save final training info
            training_info = {
                'best_epoch': best_epoch,
                'best_val_accuracies': best_val_acc,
                'train_losses': train_losses,
                'validation_accuracies': validation_accuracies,
                'learning_rates': learning_rates
            }
            info_path = os.path.join(save_dir, 'training_info.pth')
            torch.save(training_info, info_path)
 
            return train_losses, validation_accuracies
        
    
    def create_feature_extractor(self):
        """Creates a feature extractor model (everything except the last layer)"""
        # Create a new instance of the model
        feature_extractor = ConvNet()
        # Copy all weights from the trained model
        feature_extractor.load_state_dict(self.model.state_dict())
        # Replace the last layer with an Identity layer that just passes the features through
        feature_extractor.fc3 = nn.Identity()
        return feature_extractor

            
    def save_checkpoint(self, filename, epoch, loss, accuracy):
        """Save model checkpoint with all relevant training information"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'loss': loss,
            'accuracy': accuracy
        }
        torch.save(checkpoint, filename)
        logging.info(f'Checkpoint saved to {filename}')


In [ ]:
def get_device():
        if torch.backends.mps.is_available():
            device = torch.device("mps")
            logging.info(f'Using device: MPS (Apple Silicon GPU)')
        elif torch.cuda.is_available():
            device = torch.device("cuda")
            logging.info(f'Using device: CUDA GPU')
        else:
            device = torch.device("cpu")
            logging.info(f'Using device: CPU')
        return device

In [ ]:
class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        # Initialize temperature parameter T=1
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits):
        # Scale logits by temperature
        return logits / self.temperature

In [ ]:
class ECE: 
    @staticmethod

    def fit_temperature(model, logits, labels, valid_loader, device):
        model.eval()
        # Initialize temperature module
        temp_model = TemperatureScaling().to(device)

        # Minimize NLL
        optimizer = optim.LBFGS([temp_model.temperature], lr=0.01, max_iter=50)

        def closure():
            optimizer.zero_grad()
            scaled_logits = temp_model(logits)
            loss = F.cross_entropy(scaled_logits, labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        print("Optimal temperature:", temp_model.temperature.item())
        scaled_logits = temp_model(logits.to(device))  
        return scaled_logits

    def compute_ece(logits, labels, n_bins, num_classes):
        ece_metric = MulticlassCalibrationError(
        num_classes=num_classes, 
        n_bins=n_bins, 
        norm='l1'
        )
        ece_value = ece_metric(logits, labels) 
        print("ECE:", ece_value.item())


In [ ]:
def main():
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    device = get_device()

    try:
        # Load data
        data_path = 'Test_Data'  # Replace with your actual data path
        train_loader, val_loader, test_loader, classes = DatasetManager.load_data(data_path)
        logging.info(f'Loaded dataset with {Config().num_classes} classes')
       
        # Initialize model
        model = ConvNet().to(device)
        total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"number of trainable parameters of the model: {total_trainable}")
        
        # Initialize trainer 
        trainer = Trainer(model, device)
            
        # Train model
        _, validation_accuracies = trainer.train(train_loader, val_loader)
    
        # Plot training history
        plt.figure(figsize=(10, 5))
        plt.plot(validation_accuracies, label='Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('%')
        plt.legend()
        save_dir = 'visualizations'
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig('visualizations/training_curves.png')
        plt.close()

        # Add test set evaluation
        logging.info('Evaluating model on test set...')
        model.eval()

        # For storing results
        all_predictions = []
        all_labels = []
        all_filenames = []

        #For storing loits for ECE
        all_logits = []
        all_label_indices = []
        
        with torch.no_grad():
            for images, labels, paths in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                all_filenames.extend(paths)
                
                outputs, _ = model(images)
                _, predicted = torch.max(outputs.data, 1)

                # For computing ECE
                # Save logits and labels for ECE later
                all_logits.append(outputs.cpu())    
                all_label_indices.append(labels.cpu()) 

                # Convert indices to class names
                for pred, label in zip(predicted, labels):
                    all_predictions.append(classes[pred])
                    all_labels.append(classes[label])

        #Calculate Ecpected Calibration Error   
        logits = torch.cat(all_logits, dim=0) 
        label_indices = torch.cat(all_label_indices, dim=0)

        scaled_logits = ECE.fit_temperature(model, logits, label_indices, val_loader, device=torch.device("cpu"))
        ece = ECE.compute_ece(scaled_logits, label_indices, n_bins=15, num_classes=len(classes))
        
        # Save predictions of the test dataset
        df_predictions = pd.DataFrame({
            'image': all_filenames,
            'true_label': all_labels,
            'predicted_label': all_predictions
            })
        df_predictions.to_csv('predictions.csv', index=False)
        logging.info("All predictions saved as predictions.csv.")

        # Get unique combinations
        unique_combinations = sorted(list(set(all_predictions + all_labels)))
        
        # Calculate metrics using sklearn
        report = classification_report(
            all_labels,
            all_predictions,
            labels=unique_combinations,
            zero_division=0,
            output_dict=True
        )
        
        # Print metrics table
        logging.info('\nMetrics Summary:')
        logging.info('\n{:<15} {:<12} {:<12} {:<12} {:<12}'.format('Class', 'Precision', 'Recall', 'F1-Score', 'Support'))
        logging.info('-' * 63)

        # Print per-class metrics
        for combo in unique_combinations:
            metrics = report[combo]
            logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
                combo,
                metrics['precision'],
                metrics['recall'],
                metrics['f1-score'],
                metrics['support']
            ))
        
        # Print average metrics
        logging.info('-' * 63)
        logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
            'Macro Avg.',
            report['macro avg']['precision'],
            report['macro avg']['recall'],
            report['macro avg']['f1-score'],
            report['macro avg']['support']
        ))
        logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
            'Weighted Avg.',
            report['weighted avg']['precision'],
            report['weighted avg']['recall'],
            report['weighted avg']['f1-score'],
            report['weighted avg']['support']
        ))
        
        # Create confusion matrix
        cm = confusion_matrix(all_labels, all_predictions, labels=unique_combinations)
        
        # Plot confusion matrix
        plt.figure(figsize=(12, 10))
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=unique_combinations
        )
        disp.plot(xticks_rotation=90)
        plt.title('Confusion Matrix')
        plt.tight_layout()
        save_dir = 'visualizations'
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig('visualizations/confusion_matrix.png')
        plt.close()

        

    except Exception as e:
        logging.error(f'An error occurred: {str(e)}')
        raise
    finally:
        torch.cuda.empty_cache()
 
if __name__ == '__main__':
    main()